# SQL Alchemy Demo

In [ ]:
from __future__ import annotations

import os
from sqlalchemy import create_engine, text, select
from sqlalchemy.orm import Session
from sqlalchemy.exc import SQLAlchemyError, IntegrityError
from model.Klient import Klient
from model.Schuld import Schuld
from model.Glaeubiger import Glaeubiger
from model.User import User
from model.Vertrag import Vertrag

engine = create_engine(
    f'mysql+mysqlconnector://{os.getenv("DB_USER")}:{os.getenv("DB_PASSWORD")}@{os.getenv("DB_HOST")}/{os.getenv("DB_NAME")}',
    echo=True
)

### Datenbankverbindung via Engine

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT Vname, Nname FROM Klient"))
    
rows = result.fetchall()
print(rows)

### SQL Injection despite ORM

In [ ]:
# SQL Injection with textual SQL
user_id = '28 OR 1=1'
query = f"SELECT * FROM User WHERE UserId = {user_id}"

with Session(engine) as session:
    result = session.execute(text(query))
    for row in result:
        print(row)

In [ ]:
# SQLAlchemy impedes SQL injection with full ORM-based implementation
stmt = select(User).where(User.userId == user_id)
result = session.execute(stmt).scalar_one_or_none()

print(result.username, result.passwort)

### Unterschied zwischen execute() und scalars()

In [ ]:
with Session(engine) as session:
    result_1 = session.execute(select(Klient.vname, Klient.nname))

    for x in result_1:
        print(x)

In [ ]:
result_2 = session.scalars(select(Klient.nname, Klient.vname)).all()
for y in result_2:
    print(y)

### scalars() mit ganzem Objekt

In [ ]:
statement_1 = select(Klient).filter(Klient.klientenNr > 40000)
klienten_objekte = session.scalars(statement_1).all() # returns a list of Klient objects
for x in klienten_objekte:
    print(x)

In [ ]:
for x in klienten_objekte:
    print(f"Klient: {x.vname} {x.nname}")

### Objektinstanz in DB einfügen

In [ ]:
with Session(engine) as session:
    try:
        klient_obj = Klient(
            klientenNr=38201, 
            nname="Barrio", 
            vname="Eduardo", 
            strasse="Emmersmatt", 
            plz=3414, 
            ort="Rüdtligen-Alchenflüh", 
            telefonNr="0317511234", 
            beruf="arbeitslos", 
            monatseinkommen=0
            )

        session.add(klient_obj)
        session.flush()  # Flush the session to check for integrity errors before committing
    except IntegrityError as e:
        print(f"Object already exists in database: {e}")
        session.rollback()
    except SQLAlchemyError as e:
        print(f"An error occurred: {e}")
        session.rollback()
    else:
        session.commit()

### Lazy Loading & Repository Pattern

In [ ]:
from repository.schuld_repo import SchuldRepository

with Session(engine) as session:
    sRepo = SchuldRepository(session)
    schuld = sRepo.getSchuldBySchuldNr(9547)
    print(f"SchuldNr: {schuld.schuldNr}, Höhe: {schuld.hoehe}, GläubigerNr: {schuld.glaeubigerNr}, KlientenNr: {schuld.klientenNr}, Grund: {schuld.grund}")

    print("--------------------------------------------------------------------------------------------------------------------------------------------")
    schuld.glaeubiger  # Accessing the related Glaeubiger object
    print(f"Glaeubiger: {schuld.glaeubiger.vname} {schuld.glaeubiger.nname}")

### Manual Schema Generation with SQLAlchemy

In [ ]:
# Alle Tabellen müssen importiert sein, damit sie in Metadata zu finden sind

from model.Base import Base
from sqlalchemy import inspect


inspector = inspect(engine)

tables = inspector.get_table_names()
if tables:
    for t in tables:
        print(f"Table: {t}")
else:
    print("No tables found in the database.")

Base.metadata.create_all(engine)
